# 05 · Quality, single-speaker check, filter

* **quality**: DNSMOS (SIG / BAK / OVRL, P.808) and UTMOS for every clip.
* **speaker_check** (NeMo env): TitaNet embeddings of 3 s windows; `window_similarity` = the worst window's similarity to the clip's own voice. A second voice anywhere in the clip pulls it down.
* **filter**: thresholds from `configs/config.yaml` → only these clips get transcribed. Scores are cached, so tune the thresholds here and re-run `filter` (seconds).

In [ ]:
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)
if Path("/workspace").exists():
    os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
import numpy as np, pandas as pd
import plotly.express as px
from egyspeech import nb
cfg, lay = nb.cfg_and_layout()          # configs/config.yaml
print("work dir:", lay.root)

In [ ]:
nb.run("quality")

In [ ]:
nb.run("speaker_check")

In [ ]:
clips = nb.clips_frame(lay)
f = cfg.filter
for col, thr in [("dnsmos_ovrl", f.min_dnsmos_ovrl), ("dnsmos_sig", f.min_dnsmos_sig), ("dnsmos_bak", f.min_dnsmos_bak), ("utmos", f.min_utmos), ("window_similarity", f.min_window_similarity)]:
    fig = px.histogram(clips, x=col, nbins=60, title=f"{col} (threshold {thr}, below: {(clips[col] < thr).mean():.1%})")
    fig.add_vline(x=thr, line_color="red"); fig.show()

In [ ]:
print("lowest window similarity (likely two voices):")
for _, r in clips.nsmallest(3, "window_similarity").iterrows():
    nb.play(r.path, f"`{r.id}` window_similarity={r.window_similarity:.2f}")
print("lowest DNSMOS OVRL:")
for _, r in clips.nsmallest(2, "dnsmos_ovrl").iterrows():
    nb.play(r.path, f"`{r.id}` OVRL={r.dnsmos_ovrl:.2f} UTMOS={r.utmos:.2f}")
print("best clips:")
for _, r in clips.nlargest(2, "utmos").iterrows():
    nb.play(r.path, f"`{r.id}` OVRL={r.dnsmos_ovrl:.2f} UTMOS={r.utmos:.2f}")

In [ ]:
nb.run("filter")

In [ ]:
print(json.dumps(json.loads(lay.filter_report.read_text()), indent=1))